# Hangman Game — Transformer-Based Solver

## Overview

The goal of this project is to build an algorithm that plays the game of Hangman via an API.

**How the game works:** The server selects a secret word at random and returns a row of underscores (space-separated) — one for each letter. The algorithm guesses one letter at a time. If the letter is in the word, all instances are revealed in their correct positions. If not, it counts as an incorrect guess. The game ends when either (1) all letters have been correctly guessed, or (2) six incorrect guesses have been made.

**The approach:** A `guess` function takes the current masked word (with underscores) as input and returns a letter. The algorithm is trained on a dictionary of ~250,000 words but evaluated on an entirely disjoint set of ~250,000 words — so it must generalize beyond memorization.

**Baseline:** A simple frequency-matching strategy (match the masked pattern against all dictionary words, guess the most common remaining letter) achieves roughly 18% accuracy. The Transformer-based solver implemented here reaches **56% accuracy** on 1,000 recorded games.


## What I Tried
- **BPE merges:** 100, 200, 500, 750, 2000  
  All settings converged to nearly identical performance—100 merges chosen to save compute/time.
- **Classifier depth & embedding dimensions:**  
  Tested different number of layers and different dims. No consistent improvements (compute constrained so couldn't implement a full grid search).
- **Attention heads:** 4 → 8 heads  
  Similar performance; model seemed saturated.
- **LoRA fine-tuning:**  
  Injected low-rank adapters and trained for multiple epochs—no measurable gain.
- **Char-level CNN front-end:**  
  Added 1D convolutions over characters feeding into the Transformer—performance unchanged.
- **Monolingual (English-only) training:**  
  Filtering to English words degraded overall win rate.
- **Extended training:**  
  Original model from 5 → 10 epochs on full dataset; win rate rose from 55.6% → 56%.

## What I Ended Up Doing
- **Final architecture:**  
  Original architecture kept, 100 BPE merges, trained for 5 additional epochs on the full dataset.
- **Dropped modifications:**  
  LoRA adapters and CNN front-end didn’t improve performance under compute/time limits and the modifications that did improve did it only slightly after 2 epochs (not enough time and compute to retrain over 10 epochs to test and compare performace).
- **Data strategy:**  
  With no more real data available, I focused on maximizing epochs and sticking with the baseline.

## Why I Tried Each Component
- **BPE merges:**  
  To trade off vocabulary size vs. subword granularity and reduce sequence length.
- **Depth & embedding dims:**  
  To increase model capacity for capturing complex letter and semantic patterns.
- **Attention heads:**  
  To enable the model to attend to multiple letter positions simultaneously.
- **LoRA:**  
  To cheaply adapt a large pretrained core without full fine-tuning—ideal under limited compute.
- **CNN:**  
  To capture local n-gram/morphological cues before contextual encoding.
- **English-only filter:**  
  To test if specializing on one language boosted in-word semantic learning.
- **Longer training:**  
  To see if more epochs on the same data would yield diminishing or continued returns.


## Data Scarcity: The Core Bottleneck
All architectural and fine-tuning tweaks (BPE merges, deeper/wider transformers, extra heads, LoRA adapters, char-CNN front end) yielded only marginal gains. With a fixed dataset size, the model quickly plateaued around a 56 % win rate (although in practice mode it reached 59% accuracy on a run of 1000 words so we can also conclude that there is variability of performance on the test set as well). This firmly points to **data scarcity** as the primary limiter.


## Benefits of More Data
- **New Subword Patterns:** Non-English words introduce character sequences and morphological rules that don’t appear in English alone.  
- **Regularization Effect:** The “noise” from varied orthographies forces the model to learn more generalizable representations.  
- **Performance Lift:** Training on the full corpus consistently outperformed the English-only version, demonstrating that **diversity** is a form of augmentation.


## Generating Synthetic Data
To mimic the gains from diversity without collecting more real data, we can programmatically expand our corpus:

1. **Morphological Augmentation**  
   - Derive inflected or derived forms (e.g., `run → running → rerun`, `post + office → postoffice`).

2. **Noisy Spellings**  
   - Apply small random edits (swap, insert, delete letters at a 1–2% rate).  

3. **LM-Guided Mask Completion**  
   - Prompt a large pretrained language model with patterns like `_a_h_an` to harvest its top guesses as new words.


## Next Steps: Longer Training on Expanded Data
1. **Retrain** the current best architecture on the **augmented dataset**.  
2. **Scale Epochs** from 10 → 15 or 20 now that overfitting risk is reduced by data diversity but add **early stopping** just in case.  
3. **Monitor** for continued lift; if gains persist, explore further synthetic pipelines or minor hyperparameter tuning.

**Bottom line:** _Data is the lever._ By synthesizing high-quality, diverse hangman puzzles and then training longer, I should be able to push well past the current plateau (in theory).

In [1]:
import json
import requests
import random
import string
import secrets
import time
import re
import collections
import numpy as np
import torch
import torch.nn as nn

try:
    from urllib.parse import parse_qs, urlencode, urlparse
except ImportError:
    from urlparse import parse_qs, urlparse
    from urllib import urlencode

from requests.packages.urllib3.exceptions import InsecureRequestWarning

requests.packages.urllib3.disable_warnings(InsecureRequestWarning)

/Users/apple/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
with open("../data/tokenizer.json", "r") as f:
    tokenizer_dict = json.load(f)
id_to_token = {v: k for k, v in tokenizer_dict.items()}
MAX_LENGTH = 20

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LENGTH):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]
    
class HangmanTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=4, dim_feedforward=512, dropout=0.1):
        super(HangmanTransformer, self).__init__()

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.positional_encoding = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.wrong_guesses_proj = nn.Linear(26, d_model)

        self.pooling = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 26)
        )

    def forward(self, input_ids, attention_mask, wrong_guesses):

        embedded = self.token_embedding(input_ids)

        embedded = self.positional_encoding(embedded)

        encoded = self.transformer_encoder(embedded, src_key_padding_mask=~attention_mask.bool())

        encoded_transposed = encoded.transpose(1, 2)
        pooled = self.pooling(encoded_transposed).squeeze(-1)

        wrong_guesses_embedding = self.wrong_guesses_proj(wrong_guesses)

        combined = torch.cat([pooled, wrong_guesses_embedding], dim=1)

        logits = self.classifier(combined)

        return logits

In [4]:

checkpoint = torch.load("../models/10_epochs.pt", map_location=torch.device('cpu'))
model = HangmanTransformer(vocab_size=checkpoint['vocab_size'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()  

HangmanTransformer(
  (token_embedding): Embedding(127, 256)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (wrong_guesses_proj): Linear(in_features=26, out_features=256, bias=True)
  (pooling): AdaptiveAvgPool1d(output_size=1)
  (classifier): Sequential(
    (0): Linear(in_feature

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

HangmanTransformer(
  (token_embedding): Embedding(127, 256)
  (positional_encoding): PositionalEncoding()
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-3): 4 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (wrong_guesses_proj): Linear(in_features=26, out_features=256, bias=True)
  (pooling): AdaptiveAvgPool1d(output_size=1)
  (classifier): Sequential(
    (0): Linear(in_feature

In [6]:
def encode(text, max_length=20):
    if not isinstance(text, str):
        text = str(text)
    token_ids = [tokenizer_dict.get(char, tokenizer_dict.get("<unk>", 0)) for char in text]
    if len(token_ids) < max_length:
        token_ids = token_ids + [tokenizer_dict.get("<pad>", 0)] * (max_length - len(token_ids))
    else:
        token_ids = token_ids[:max_length]
    return token_ids

def make_guess(model, tokenizer_dict, masked_word, wrong_guesses):
    input_ids = torch.tensor([encode(masked_word)]).to(device)
    
    attention_mask = torch.ones_like(input_ids).to(device)
    pad_token_id = tokenizer_dict.get("<pad>", 0)
    attention_mask[input_ids == pad_token_id] = 0
    

    wrong_guesses_tensor = torch.zeros(1, 26).to(device)
    alphabet = list('abcdefghijklmnopqrstuvwxyz')
    for letter in wrong_guesses:
        if letter in alphabet:
            idx = alphabet.index(letter)
            wrong_guesses_tensor[0, idx] = 1
            
    with torch.no_grad():
        logits = model(input_ids, attention_mask, wrong_guesses_tensor)
        probabilities = torch.sigmoid(logits)[0]
        
    _, indices = torch.sort(probabilities, descending=True)
    for idx in indices:
        guess = alphabet[idx.item()]
        if guess not in wrong_guesses and guess not in masked_word:
            return guess
    return None

In [7]:
HANGMAN_API_URL = "http://localhost:5000/hangman"

class HangmanAPI(object):
    def __init__(self, access_token=None, session=None, timeout=None):
        self.hangman_url = self.determine_hangman_url()
        self.access_token = access_token
        self.session = session or requests.Session()
        self.timeout = timeout
        self.guessed_letters = []
        self.max_seq_len = 20
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        full_dictionary_location = "../data/words_250000_train.txt"
        self.full_dictionary = self.build_dictionary(full_dictionary_location)        
        self.full_dictionary_common_letter_sorted = collections.Counter("".join(self.full_dictionary)).most_common()
        
        self.current_dictionary = []
        
    @staticmethod
    def determine_hangman_url():
        # Set your own Hangman API endpoint here
        return HANGMAN_API_URL

    def guess(self, word): 
        word = word.split(' ')
        word = ''.join(word)  
        input_ids = torch.tensor([encode(word)]).to(device)
        wrong_guesses = [i for i in self.guessed_letters if i not in word]
        if len(word) == 3:
            dictionary_subset = [w for w in self.full_dictionary if len(w) == len(word)]
            letter_counts = collections.Counter("".join(dictionary_subset))
            sorted_letters = [letter for letter, count in letter_counts.most_common()]
            for letter in sorted_letters:
                if letter not in wrong_guesses and letter not in word:
                    return letter
        if len(word) == 4:
            dictionary_subset = [w for w in self.full_dictionary if len(w) == len(word)]
            letter_counts = collections.Counter("".join(dictionary_subset))
            sorted_letters = [letter for letter, count in letter_counts.most_common()]
            for letter in sorted_letters:
                if letter not in wrong_guesses and letter not in word:
                    return letter
        attention_mask = torch.ones_like(input_ids).to(device)
        pad_token_id = tokenizer_dict.get("<pad>", 0)
        attention_mask[input_ids == pad_token_id] = 0
        
        wrong_guesses_tensor = torch.zeros(1, 26).to(device)
        alphabet = list('abcdefghijklmnopqrstuvwxyz')
        for letter in wrong_guesses:
            if letter in alphabet:
                idx = alphabet.index(letter)
                wrong_guesses_tensor[0, idx] = 1
                
 
        with torch.no_grad():
            logits = model(input_ids, attention_mask, wrong_guesses_tensor)
            probabilities = torch.sigmoid(logits)[0]
            

        _, indices = torch.sort(probabilities, descending=True)
        for idx in indices:
            guess = alphabet[idx.item()]
            if guess not in wrong_guesses and guess not in word:
                return guess
        
    def build_dictionary(self, dictionary_file_location):
        text_file = open(dictionary_file_location,"r")
        full_dictionary = text_file.read().splitlines()
        text_file.close()
        return full_dictionary
                
    def start_game(self, practice=True, verbose=True):
        # reset guessed letters to empty set and current plausible dictionary to the full dictionary
        self.guessed_letters = []
        self.current_dictionary = self.full_dictionary
                         
        response = self.request("/new_game", {"practice":practice})
        if response.get('status')=="approved":
            game_id = response.get('game_id')
            word = response.get('word')
            tries_remains = response.get('tries_remains')
            if verbose:
                print("Successfully start a new game! Game ID: {0}. # of tries remaining: {1}. Word: {2}.".format(game_id, tries_remains, word))
            while tries_remains>0:
                # get guessed letter from user code
                guess_letter = self.guess(word)
                    
                # append guessed letter to guessed letters field in hangman object
                self.guessed_letters.append(guess_letter)
                if verbose:
                    print("Guessing letter: {0}".format(guess_letter))
                    
                try:    
                    res = self.request("/guess_letter", {"request":"guess_letter", "game_id":game_id, "letter":guess_letter})
                except HangmanAPIError:
                    print('HangmanAPIError exception caught on request.')
                    continue
                except Exception as e:
                    print('Other exception caught on request.')
                    raise e
               
                if verbose:
                    print("Sever response: {0}".format(res))
                status = res.get('status')
                tries_remains = res.get('tries_remains')
                if status=="success":
                    if verbose:
                        print("Successfully finished game: {0}".format(game_id))
                    return True
                elif status=="failed":
                    reason = res.get('reason', '# of tries exceeded!')
                    if verbose:
                        print("Failed game: {0}. Because of: {1}".format(game_id, reason))
                    return False
                elif status=="ongoing":
                    word = res.get('word')
        else:
            if verbose:
                print("Failed to start a new game")
        return status=="success"
        
    def my_status(self):
        return self.request("/my_status", {})
    
    def request(
            self, path, args=None, post_args=None, method=None):
        if args is None:
            args = dict()
        if post_args is not None:
            method = "POST"

        # Add `access_token` to post_args or args if it has not already been
        # included.
        if self.access_token:
            # If post_args exists, we assume that args either does not exists
            # or it does not need `access_token`.
            if post_args and "access_token" not in post_args:
                post_args["access_token"] = self.access_token
            elif "access_token" not in args:
                args["access_token"] = self.access_token

        time.sleep(0.2)

        num_retry, time_sleep = 50, 2
        for it in range(num_retry):
            try:
                response = self.session.request(
                    method or "GET",
                    self.hangman_url + path,
                    timeout=self.timeout,
                    params=args,
                    data=post_args,
                    verify=False
                )
                break
            except requests.HTTPError as e:
                response = json.loads(e.read())
                raise HangmanAPIError(response)
            except requests.exceptions.SSLError as e:
                if it + 1 == num_retry:
                    raise
                time.sleep(time_sleep)

        headers = response.headers
        if 'json' in headers['content-type']:
            result = response.json()
        elif "access_token" in parse_qs(response.text):
            query_str = parse_qs(response.text)
            if "access_token" in query_str:
                result = {"access_token": query_str["access_token"][0]}
                if "expires" in query_str:
                    result["expires"] = query_str["expires"][0]
            else:
                raise HangmanAPIError(response.json())
        else:
            raise HangmanAPIError('Maintype was not text, or querystring')

        if result and isinstance(result, dict) and result.get("error"):
            raise HangmanAPIError(result)
        return result
    
class HangmanAPIError(Exception):
    def __init__(self, result):
        self.result = result
        self.code = None
        try:
            self.type = result["error_code"]
        except (KeyError, TypeError):
            self.type = ""

        try:
            self.message = result["error_description"]
        except (KeyError, TypeError):
            try:
                self.message = result["error"]["message"]
                self.code = result["error"].get("code")
                if not self.type:
                    self.type = result["error"].get("type", "")
            except (KeyError, TypeError):
                try:
                    self.message = result["error_msg"]
                except (KeyError, TypeError):
                    self.message = result

        Exception.__init__(self, self.message)

# API Usage Examples

## To start a new game:
1. Make sure you have implemented your own `guess` method.
2. Set your `access_token` and create a `HangmanAPI` object.
3. Start a game by calling the `start_game` method.
4. To test without recording results, set `practice=1`.
5. Note: There is a rate limit of 20 new games per minute.

In [8]:
api = HangmanAPI(access_token="YOUR_ACCESS_TOKEN", timeout=2000)

## Playing practice games:
Run the cell below to play practice games (up to 100,000).

In [ ]:
api.start_game(practice=1,verbose=True)
[total_practice_runs,total_recorded_runs,total_recorded_successes,total_practice_successes] = api.my_status() # Get my game stats: (# of tries, # of wins)
practice_success_rate = total_practice_successes / total_practice_runs if total_practice_runs > 0 else 0
print('run %d practice games out of an allotted 100,000. practice success rate so far = %.3f' % (total_practice_runs, practice_success_rate))

## Playing recorded games:
Finalize your code before running the cell below. Recorded games count toward your final score and cannot be re-run.

In [9]:
for i in range(1000):
    print('Playing ', i, ' th game')
    api.start_game(practice=0,verbose=False)
    
    # DO NOT REMOVE as otherwise the server may lock you out for too high frequency of requests
    time.sleep(0.5)

Playing  0  th game


/Users/apple/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/transformer.py:384: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:179.)
  output = torch._nested_tensor_from_mask(output, src_key_padding_mask.logical_not(), mask_check=False)


Playing  1  th game
Playing  2  th game
Playing  3  th game
Playing  4  th game
Playing  5  th game
Playing  6  th game
Playing  7  th game
Playing  8  th game
Playing  9  th game
Playing  10  th game
Playing  11  th game
Playing  12  th game
Playing  13  th game
Playing  14  th game
Playing  15  th game
Playing  16  th game
Playing  17  th game
Playing  18  th game
Playing  19  th game
Playing  20  th game
Playing  21  th game
Playing  22  th game
Playing  23  th game
Playing  24  th game
Playing  25  th game
Playing  26  th game
Playing  27  th game
Playing  28  th game
Playing  29  th game
Playing  30  th game
Playing  31  th game
Playing  32  th game
Playing  33  th game
Playing  34  th game
Playing  35  th game
Playing  36  th game
Playing  37  th game
Playing  38  th game
Playing  39  th game
Playing  40  th game
Playing  41  th game
Playing  42  th game
Playing  43  th game
Playing  44  th game
Playing  45  th game
Playing  46  th game
Playing  47  th game
Playing  48  th game
P

## Game statistics
Call `my_status()` to get the total number of games played and wins.

In [10]:
[total_practice_runs,total_recorded_runs,total_recorded_successes,total_practice_successes] = api.my_status() # Get my game stats: (# of tries, # of wins)
success_rate = total_recorded_successes/total_recorded_runs
print('overall success rate = %.3f' % success_rate)

overall success rate = 0.560
